# Week 4：文本表示与聚类比较

## 实验目的

使用与第二周相同的 20 Newsgroups 四类别文本子集，
比较 TF-IDF 与预训练 Sentence Transformer 文本向量的聚类效果。

两种表示均使用 K-means，聚类数设为 K = 4。

评价指标：
- Adjusted Rand Index（ARI）
- Normalized Mutual Information（NMI）
- Silhouette score（轮廓系数）

真实类别标签仅用于聚类完成后的评价，不参与聚类训练。

## 实验流程

1. 检查 Python 环境。
2. 加载并清理文本数据。
3. 构建 TF-IDF 表示并进行聚类。
4. 使用预训练模型生成文本向量并进行聚类。
5. 比较结果并记录观察。

In [1]:
#检查1环境
import os
import sys
from importlib.metadata import version, PackageNotFoundError

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])

packages = [
    "numpy",
    "pandas",
    "scikit-learn",
    "torch",
    "sentence-transformers",
    "matplotlib",
    "threadpoolctl",
]

for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

Python executable: /opt/anaconda3/bin/python
Python version: 3.13.9
numpy: 2.3.5
pandas: 2.3.3
scikit-learn: 1.7.2
torch: 2.14.0
sentence-transformers: 6.2.0
matplotlib: 3.10.6
threadpoolctl: 3.5.0


In [3]:
%pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 8.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 26.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 46.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 29.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 26.5 MB/s  0:00:00
  Attempting uninstall: regex
    Found existing installation: regex 2025.9.1
    Uninstalling regex-2025.9.1:
      Successfully uninstalled regex-2025.9.1
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
  Attempting uninstall: hf-xet0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/11 [httpcore2]
    Found existing installation: hf-xet 1.5.0━━━━━━━━━━━━━━━━━  3/11 [httpcore2]
    Uninstalling hf-xet-1.5.0:90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/11 [httpcore2]
      Successfully uninstalled hf-xet-1.5.0━━━━━━━━━━━━━━━━━━━  3/11 

In [2]:
from sentence_transformers import SentenceTransformer

print("SentenceTransformer imported successfully.")

W1105 09:21:07.902000 5401 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


SentenceTransformer imported successfully.


## 1. 加载与清理文本数据

使用 20 Newsgroups 数据集中的四个类别：
- comp.graphics
- sci.space
- alt.atheism
- talk.religion.misc

沿用第二周的设置：
- 使用 subset="all"。
- 移除文章头部、签名尾部和引用内容。
- 删除清理后只包含空白字符的文档。
- 固定随机种子为 42。

TF-IDF 和 Sentence Transformer 将使用相同的文档及顺序。
真实类别标签仅用于后续评价，不参与聚类训练。

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

RANDOM_SEED = 42

categories = [
    "comp.graphics",
    "sci.space",
    "alt.atheism",
    "talk.religion.misc",
]

newsgroups = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=RANDOM_SEED,
)#读取数据

text_df = pd.DataFrame({
    "text": newsgroups.data,
    "label": newsgroups.target,
})

label_to_category = dict(enumerate(newsgroups.target_names))
text_df["category"] = text_df["label"].map(label_to_category)

empty_mask = text_df["text"].str.strip().eq("")

clean_text_df = (
    text_df.loc[~empty_mask]
    .copy()
    .reset_index(names="original_index")
)

documents = clean_text_df["text"].tolist()
true_labels = clean_text_df["label"].to_numpy()

assert len(documents) == len(true_labels)
assert all(document.strip() for document in documents)

print("Original document count:", len(text_df))
print("Removed empty documents:", int(empty_mask.sum()))
print("Remaining document count:", len(documents))
print("Label shape:", true_labels.shape)

print("\nLabel mapping:", label_to_category)
print("\nDocuments per category:")
print(clean_text_df["category"].value_counts().sort_index())

print("\nFirst document preview:")
print(documents[0][:500])

Original document count: 3387
Removed empty documents: 92
Remaining document count: 3295
Label shape: (3295,)

Label mapping: {0: 'alt.atheism', 1: 'comp.graphics', 2: 'sci.space', 3: 'talk.religion.misc'}

Documents per category:
category
alt.atheism           779
comp.graphics         955
sci.space             955
talk.religion.misc    606
Name: count, dtype: int64

First document preview:
My point is that you set up your views as the only way to believe.  Saying 
that all eveil in this world is caused by atheism is ridiculous and 
counterproductive to dialogue in this newsgroups.  I see in your posts a 
spirit of condemnation of the atheists in this newsgroup bacause they don'
t believe exactly as you do.  If you're here to try to convert the atheists 
here, you're failing miserably.  Who wants to be in position of constantly 
defending themselves agaist insulting attacks, like y


## 2. 构建 TF-IDF 表示

沿用第二周的 TF-IDF 参数：
- 将文本转换为小写。
- 使用英文停用词表。
- 仅保留至少出现在 2 篇文档中的词。
- 过滤出现在超过 95% 文档中的词。
- 对每篇文档的非零向量进行 L2 归一化。

TF-IDF 矩阵的每一行对应一篇文章，每一列对应词表中的一个词。

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2,
    max_df=0.95,
    norm="l2",
)#制定文本转换规则

X_tfidf = tfidf_vectorizer.fit_transform(documents)
feature_names = tfidf_vectorizer.get_feature_names_out()

zero_vector_mask = X_tfidf.getnnz(axis=1) == 0

print("TF-IDF matrix shape:", X_tfidf.shape)
print("Number of documents:", X_tfidf.shape[0])
print("Vocabulary size:", len(feature_names))
print("Zero-vector documents:", int(zero_vector_mask.sum()))
print("Example vocabulary terms:", feature_names[:20])

TF-IDF matrix shape: (3295, 17780)
Number of documents: 3295
Vocabulary size: 17780
Zero-vector documents: 9
Example vocabulary terms: ['00' '000' '0000' '00000' '000005102000' '00041032' '0004422' '0007'
 '000usd' '00196' '0028' '0029' '0038' '0049' '006' '0094' '00index'
 '00pm' '01' '015']


`TfidfVectorizer(...)`：制定文本转换规则

| 参数 | 含义 | 小例子 |
|---|---|---|
| `lowercase=True` | 先把字母转换为小写 | `Space` 和 `space` 视为同一个词 |
| `stop_words="english"` | 过滤内置英文停用词表中的词 | 例如 `the`、`and` |
| `min_df=2` | 一个词至少出现在两篇文章中才保留 | 同一篇文章出现十次，仍只算一篇 |
| `max_df=0.95` | 过滤出现在超过 95% 文章中的词 | 这些词可能缺少区分文章的作用 |
| `norm="l2"` | 把每个非零文档向量的长度调整为 1 | 减少向量大小差异对距离比较的影响 |

## 3. TF-IDF 特征上的 K-means 聚类

使用全部 3295 篇文档的 TF-IDF 向量进行聚类。

参数设置：
- 聚类数：4
- 初始化方法：k-means++
- 初始化次数：10
- 每次运行的最大迭代次数：300
- 随机种子：42

聚类只使用文本特征，不使用真实类别标签。
保留并记录 9 篇 TF-IDF 向量为零的文档。

In [ ]:
from sklearn.cluster import KMeans
from threadpoolctl import threadpool_limits

tfidf_kmeans = KMeans(
    n_clusters=4,#把文章分成4组
    init="k-means++",
    n_init=10,#尝试 10 次不同的初始化，保留簇内平方距离总和最小的结果
    max_iter=300,#每次尝试最多迭代 300 次
    random_state=RANDOM_SEED,
)

print("Starting TF-IDF clustering...")

with threadpool_limits(limits=1):
    tfidf_clusters = tfidf_kmeans.fit_predict(X_tfidf)

print("TF-IDF clustering completed.")
print("Cluster label shape:", tfidf_clusters.shape)
print("Number of clusters:", len(np.unique(tfidf_clusters)))
print("Iterations in selected run:", tfidf_kmeans.n_iter_)

print("\nDocuments per cluster:")
print(pd.Series(tfidf_clusters).value_counts().sort_index())

Starting TF-IDF clustering...
TF-IDF clustering completed.
Cluster label shape: (3295,)
Number of clusters: 4
Iterations in selected run: 36

Documents per cluster:
0    1837
1     452
2     321
3     685
Name: count, dtype: int64


## 4. 评价 TF-IDF 聚类结果

使用三个指标：
- ARI：比较聚类分组与真实类别，并校正随机一致性的影响。
（两篇文章是否应该分在一起？）
- NMI：衡量聚类分组与真实类别之间共享的信息。
（如果某个簇中的文章大多来自一个类别，则这个簇编号就能提供一些类别信息）
- Silhouette score：根据特征空间中的距离，评价簇内紧密程度与簇间分离程度。
（同一簇内部文章是否靠近，不同簇文章是否分开）

ARI 和 NMI 使用真实类别标签。
Silhouette score 不使用真实类别标签。

轮廓系数使用全部 3295 篇文档，在 TF-IDF 特征空间中按欧氏距离计算。

In [7]:
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
)

tfidf_ari = adjusted_rand_score(
    true_labels,
    tfidf_clusters,
)

tfidf_nmi = normalized_mutual_info_score(
    true_labels,
    tfidf_clusters,
    average_method="arithmetic",
)

print("Calculating silhouette score...")

with threadpool_limits(limits=1):
    tfidf_silhouette = silhouette_score(
        X_tfidf,
        tfidf_clusters,
        metric="euclidean",
    )

print(f"TF-IDF ARI: {tfidf_ari:.4f}")
print(f"TF-IDF NMI: {tfidf_nmi:.4f}")
print(f"TF-IDF Silhouette (Euclidean): {tfidf_silhouette:.4f}")

Calculating silhouette score...
TF-IDF ARI: 0.1959
TF-IDF NMI: 0.3699
TF-IDF Silhouette (Euclidean): 0.0040


### TF-IDF 基线观察

在 3295 篇文档上，使用 17780 维 TF-IDF 特征和 K-means（K=4），得到：

| 指标 | 结果 |
|---|---:|
| ARI | 0.1959 |
| NMI | 0.3699 |
| Silhouette（欧氏距离） | 0.0040 |

ARI 和 NMI 表明，聚类分组与真实类别存在一定关联，但未完全对应。
轮廓系数接近零，说明在当前 TF-IDF 特征空间和欧氏距离下，
整体簇间分离不明显。

实验保留了 9 篇 TF-IDF 向量为零的文档。
这些文档无法通过当前 TF-IDF 特征提供词语区分信息，是本基线的一项局限。

接下来使用同一批文档的预训练文本向量进行聚类比较。

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("Loading pretrained model...")

sentence_model = SentenceTransformer(
    MODEL_NAME,
    device="cpu",
)

print("Model loaded successfully.")
print("Model name:", MODEL_NAME)
print("Device:", sentence_model.device)
print("Embedding dimension:", sentence_model.get_embedding_dimension())
print("Maximum sequence length:", sentence_model.max_seq_length)

Loading pretrained model...


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

... (2 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully.
Model name: sentence-transformers/all-MiniLM-L6-v2
Device: cpu
Embedding dimension: 384
Maximum sequence length: 256


/var/folders/27/k_4b10p53mzchh2x9bgjs4680000gn/T/ipykernel_5401/1328129868.py:15: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", sentence_model.get_sentence_embedding_dimension())


## 5. 检查预训练文本向量

使用 sentence-transformers/all-MiniLM-L6-v2 模型，
先对三句示例文本进行编码。

每句文本转换为一个 384 维向量。
输出向量进行 L2 归一化，使每个向量的长度约为 1。

本步骤仅进行推理，不更新预训练模型的参数。
示例文本不加入正式实验数据。

In [ ]:
example_texts = [
    "The spacecraft is traveling to the moon.",
    "Astronauts are exploring space.",
    "Computer graphics can create realistic images.",
]

example_embeddings = sentence_model.encode(
    example_texts,
    batch_size=3,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Number of example texts:", len(example_texts))
print("Embedding shape:", example_embeddings.shape)

print("\nFirst 8 values of the first embedding:")
print(example_embeddings[0, :8])#取第一句文本向量的前八个分量

print("\nEmbedding lengths:")
print(np.linalg.norm(example_embeddings, axis=1))

Number of example texts: 3
Embedding shape: (3, 384)

First 8 values of the first embedding:
[ 0.03132223 -0.02986846 -0.01634685  0.02143437 -0.04338756 -0.03947887
  0.01173981 -0.01915557]

Embedding lengths:
[1. 1. 1.]


### 示例文本的余弦相似度

比较以下三句文本：

1. The spacecraft is traveling to the moon.
2. Astronauts are exploring space.
3. Computer graphics can create realistic images.

前两句都与太空有关，第三句与计算机图形学有关。
通过向量之间的余弦相似度，观察模型是否体现了这种语义关系。

由于向量已经进行 L2 归一化，
两个向量的点积等于它们的余弦相似度。

In [11]:
example_similarities = example_embeddings @ example_embeddings.T

similarity_table = pd.DataFrame(
    example_similarities,
    index=["Sentence 1", "Sentence 2", "Sentence 3"],
    columns=["Sentence 1", "Sentence 2", "Sentence 3"],
)

print("Cosine similarity matrix:")
display(similarity_table.round(4))

Cosine similarity matrix:


,Sentence 1,Sentence 2,Sentence 3
Sentence 1,1.0000,0.5011,-0.0636
Sentence 2,0.5011,1.0000,0.0340
Sentence 3,-0.0636,0.0340,1.0000


两个特点：
- 对角线应该接近 1：每句话与自己的向量方向完全相同。
- 表格应该对称：第一句与第二句的相似度，等于第二句与第一句的相似度。

## 6. 生成全部文档的文本向量

使用预训练模型 sentence-transformers/all-MiniLM-L6-v2，
对与 TF-IDF 基线相同的 3295 篇文档进行编码。

设置：
- 设备：CPU
- 批大小：32
- 输出维度：384
- 输出向量进行 L2 归一化
- 最大输入长度：256 个 token
- 不对模型进行微调

超过最大输入长度的文档会被截断，因此向量可能无法反映全文信息。

保存向量、真实标签、原始文档索引及主要编码设置，
便于后续聚类和结果追踪。

In [12]:
from pathlib import Path

print("Encoding all documents...")

X_embeddings = sentence_model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

assert X_embeddings.shape == (len(documents), 384)
assert np.isfinite(X_embeddings).all()

embedding_norms = np.linalg.norm(X_embeddings, axis=1)

output_dir = Path("week4_text_results")
output_dir.mkdir(parents=True, exist_ok=True)

embedding_path = output_dir / "sentence_embeddings.npz"

np.savez_compressed(
    embedding_path,
    embeddings=X_embeddings,
    true_labels=true_labels,
    original_indices=clean_text_df["original_index"].to_numpy(),
    model_name=MODEL_NAME,
    max_seq_length=sentence_model.max_seq_length,
    normalized=True,
)

print("Encoding completed.")
print("Embedding matrix shape:", X_embeddings.shape)
print(f"Minimum embedding length: {embedding_norms.min():.6f}")
print(f"Maximum embedding length: {embedding_norms.max():.6f}")
print("Saved to:", embedding_path.resolve())

Encoding all documents...


Batches:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding completed.
Embedding matrix shape: (3295, 384)
Minimum embedding length: 1.000000
Maximum embedding length: 1.000000
Saved to: /Users/cheeseowl/Desktop/科研/week4_text_results/sentence_embeddings.npz


## 7. 预训练文本向量上的 K-means 聚类

使用 3295 篇文档的 384 维归一化文本向量进行聚类。

沿用 TF-IDF 基线的 K-means 参数：
- 聚类数：4
- 初始化方法：k-means++
- 初始化次数：10
- 每次运行的最大迭代次数：300
- 随机种子：42

聚类只使用文本向量，真实类别标签不参与训练。

In [13]:
embedding_kmeans = KMeans(
    n_clusters=4,
    init="k-means++",
    n_init=10,
    max_iter=300,
    random_state=RANDOM_SEED,
)

print("Starting embedding clustering...")

with threadpool_limits(limits=1):
    embedding_clusters = embedding_kmeans.fit_predict(X_embeddings)

print("Embedding clustering completed.")
print("Cluster label shape:", embedding_clusters.shape)
print("Number of clusters:", len(np.unique(embedding_clusters)))
print("Iterations in selected run:", embedding_kmeans.n_iter_)

print("\nDocuments per cluster:")
print(pd.Series(embedding_clusters).value_counts().sort_index())

Starting embedding clustering...
Embedding clustering completed.
Cluster label shape: (3295,)
Number of clusters: 4
Iterations in selected run: 12

Documents per cluster:
0    891
1    864
2    787
3    753
Name: count, dtype: int64


## 8. 比较两种文本表示的聚类效果

对预训练文本向量的聚类结果计算：
- ARI
- NMI
- Silhouette score（欧氏距离）

两种表示使用相同的 3295 篇文档及 K-means 参数。
轮廓系数均使用全部文档，在各自的特征空间中计算。

ARI 和 NMI 用于比较聚类分组与真实类别的对应关系。
轮廓系数用于描述各自特征空间中的簇结构，需要结合其他指标解读。

In [14]:
embedding_ari = adjusted_rand_score(
    true_labels,
    embedding_clusters,
)

embedding_nmi = normalized_mutual_info_score(
    true_labels,
    embedding_clusters,
    average_method="arithmetic",
)

print("Calculating embedding silhouette score...")

with threadpool_limits(limits=1):
    embedding_silhouette = silhouette_score(
        X_embeddings,
        embedding_clusters,
        metric="euclidean",
    )

comparison_df = pd.DataFrame({
    "Representation": ["TF-IDF", "Sentence Transformer"],
    "Dimension": [X_tfidf.shape[1], X_embeddings.shape[1]],
    "ARI": [tfidf_ari, embedding_ari],
    "NMI": [tfidf_nmi, embedding_nmi],
    "Silhouette (Euclidean)": [tfidf_silhouette, embedding_silhouette],
})

display(comparison_df.round(4))

Calculating embedding silhouette score...


,Representation,Dimension,ARI,NMI,Silhouette (Euclidean)
0,TF-IDF,17780,0.1959,0.3699,0.0040
1,Sentence Transformer,384,0.5025,0.5306,0.0434


### 两种文本表示的初步比较

| 表示 | 维度 | ARI | NMI | Silhouette（欧氏距离） |
|---|---:|---:|---:|---:|
| TF-IDF | 17780 | 0.1959 | 0.3699 | 0.0040 |
| Sentence Transformer | 384 | 0.5025 | 0.5306 | 0.0434 |

在相同文档集合及主要 K-means 参数下，预训练文本向量的
ARI 和 NMI 均高于 TF-IDF，说明其聚类分组与真实类别更加一致。

一种可能的解释是：TF-IDF 主要根据词项及其权重表示文章，
而预训练文本模型能够利用训练中学到的语义关系，
使表达不同但主题相关的文本获得较接近的向量。
这一解释与结果相符，但本实验尚未单独验证提升的具体来源。

不能将提升简单归因于维度从 17780 降至 384。
表示方法、预训练知识、词语处理方式和输入长度限制等因素均有变化。

文本向量的轮廓系数高于 TF-IDF，但仍接近零。
由于两个分数来自不同特征空间，需要结合 ARI、NMI 和类别分布解读。

当前结果基于随机种子 42 的一次配置，
支持本次设置下的比较，不代表所有文本数据上都会得到相同结论。

In [15]:
evaluation_categories = pd.Series(
    clean_text_df["category"].to_numpy(),
    name="True category",
)

tfidf_category_table = pd.crosstab(
    pd.Series(tfidf_clusters, name="Cluster"),
    evaluation_categories,
)

embedding_category_table = pd.crosstab(
    pd.Series(embedding_clusters, name="Cluster"),
    evaluation_categories,
)

print("TF-IDF: cluster and category counts")
display(tfidf_category_table)

print("Sentence Transformer: cluster and category counts")
display(embedding_category_table)

TF-IDF: cluster and category counts


True category,alt.atheism,comp.graphics,sci.space,talk.religion.misc
Cluster,,,,
0,593,307,479,458
1,5,7,439,1
2,176,0,0,145
3,5,641,37,2


Sentence Transformer: cluster and category counts


True category,alt.atheism,comp.graphics,sci.space,talk.religion.misc
Cluster,,,,
0,364,92,186,249
1,3,826,34,1
2,12,37,731,7
3,400,0,4,349


读第二张表
| 文本向量的簇 | 主要内容 | 观察 |
|---|---|---|
| 簇 1 | 826 篇图形学文章，共 864 篇 | 主要对应图形学 |
| 簇 2 | 731 篇太空文章，共 787 篇 | 主要对应太空科学 |
| 簇 0 | 无神论 364 篇、宗教讨论 249 篇，另有其他类别 | 类别混合较明显 |
| 簇 3 | 无神论 400 篇、宗教讨论 349 篇，共 753 篇 | 以两个宗教相关类别为主，但没有将它们分开 |

按真实类别看
| 真实类别 | TF-IDF 最大集中簇包含的该类文章 | 文本向量最大集中簇包含的该类文章 |
|---|---:|---:|
| 图形学 | 641 / 955，约 67.12% | 826 / 955，约 86.49% |
| 太空科学 | 439 / 955，约 45.97% | 731 / 955，约 76.54% |

### 簇与真实类别的对应关系

TF-IDF 的簇 0 包含 1837 篇文章，四个真实类别都有较多文章落入其中，
表现出明显的类别混合。

TF-IDF 的簇 1 主要包含太空文章，簇 3 主要包含图形学文章，
但这两个真实类别仍有较多文章分散在其他簇中。

使用预训练文本向量后：
- 簇 1 包含 826 篇图形学文章。
- 簇 2 包含 731 篇太空文章。
- 图形学和太空文章分别更加集中于各自的主要簇。

但是，无神论与宗教讨论文章仍同时出现在簇 0 和簇 3，
没有形成两个分别对应真实类别的簇。
两类文本可能共享宗教、信仰等讨论内容，这是混合的一种可能解释，
具体原因仍需检查实际文章才能确认。

交叉表进一步支持 ARI、NMI 的比较结果：
在本次设置下，预训练文本向量得到的分组更符合真实类别，
但仍存在类别混合和文章分散的情况。

簇编号只用于标识分组，不能直接当作真实类别编号。

In [16]:
output_dir.mkdir(parents=True, exist_ok=True)

comparison_df.to_csv(
    output_dir / "clustering_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

tfidf_category_table.to_csv(
    output_dir / "tfidf_category_counts.csv",
    encoding="utf-8-sig",
)

embedding_category_table.to_csv(
    output_dir / "embedding_category_counts.csv",
    encoding="utf-8-sig",
)

assignments_df = clean_text_df[
    ["original_index", "label", "category"]
].copy()

assignments_df["tfidf_cluster"] = tfidf_clusters
assignments_df["embedding_cluster"] = embedding_clusters

assignments_df.to_csv(
    output_dir / "cluster_assignments.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Result tables saved.")
print("Output directory:", output_dir.resolve())

Result tables saved.
Output directory: /Users/cheeseowl/Desktop/科研/week4_text_results


## 9. 实验总结

### 实验目的

比较 TF-IDF 与预训练 Sentence Transformer 文本向量，
分析不同文本表示对文档聚类效果的影响。

### 数据与实验设置

使用 20 Newsgroups 中的四个类别：
- alt.atheism
- comp.graphics
- sci.space
- talk.religion.misc

加载 subset="all"，移除文章头部、签名尾部及引用内容。
原始数据包含 3387 篇文档，删除 92 篇空文档后，保留 3295 篇。

两种表示使用相同的文档和顺序。
真实类别标签仅用于聚类后的评价及分析，不参与聚类训练。

| 设置 | 内容 |
|---|---|
| 随机种子 | 42 |
| 聚类方法 | K-means |
| 聚类数 | 4 |
| 初始化方法 | k-means++ |
| 初始化次数 | 10 |
| 每次运行最大迭代次数 | 300 |
| 运行线程限制 | 1 |
| 评价指标 | ARI、NMI、Silhouette score |
| 轮廓系数设置 | 全部文档、欧氏距离 |

本实验在这 3295 篇文档上构建表示、拟合聚类并评价分组，
没有另外划分用于检验新文档聚类表现的测试集。

### 两种文本表示

#### TF-IDF

参数为：
- lowercase=True
- stop_words="english"
- min_df=2
- max_df=0.95
- norm="l2"

得到形状为 (3295, 17780) 的稀疏矩阵。
每一列对应词表中的一个词项。

有 9 篇非空文档在词语过滤后变成全零向量。
本实验保留这些文档，并将其作为 TF-IDF 基线的一项局限记录。

#### Sentence Transformer

使用 sentence-transformers/all-MiniLM-L6-v2 预训练模型。

设置为：
- 设备：CPU
- 批大小：32
- 输出维度：384
- 输出向量进行 L2 归一化
- 最大输入长度：256 个 token

得到形状为 (3295, 384) 的稠密矩阵。

直接使用预训练模型进行编码，没有进行微调或参数更新，
因此本部分没有模型训练损失需要记录。

### 聚类结果

| 表示 | 维度 | ARI | NMI | Silhouette（欧氏距离） |
|---|---:|---:|---:|---:|
| TF-IDF | 17780 | 0.1959 | 0.3699 | 0.0040 |
| Sentence Transformer | 384 | 0.5025 | 0.5306 | 0.0434 |

相较于 TF-IDF，文本向量的 ARI 提高 0.3066，
NMI 提高 0.1607。

这说明在本次设置下，文本向量得到的聚类分组与真实类别更加一致。
ARI 和 NMI 是聚类评价分数，不能解释为分类准确率。

文本向量的轮廓系数也更高，但仍接近零。
两个轮廓系数来自不同特征空间，需要结合其他指标解读，
不能据此认为四个类别已经完全分离。

### 类别分布观察

TF-IDF 的簇 0 包含 1837 篇文档，四个真实类别均有较多文章落入其中，
说明存在明显的类别混合。

使用文本向量后，图形学和太空文章更加集中于各自的主要簇：

| 真实类别 | TF-IDF 最大集中簇中的该类文章 | 文本向量最大集中簇中的该类文章 |
|---|---:|---:|
| comp.graphics | 641 / 955 | 826 / 955 |
| sci.space | 439 / 955 | 731 / 955 |

但是，无神论与宗教讨论文章仍同时分布在文本向量的簇 0 和簇 3，
没有形成分别对应这两个类别的簇。

这两个类别可能共享部分讨论内容，但具体混合原因还需要阅读文章确认。

### 为什么不同表示会影响聚类？

K-means 根据特征空间中的距离进行分组。
改变文本表示，就会改变文章之间的距离关系，从而影响聚类结果。

TF-IDF 主要利用词项及其权重表示文章。
预训练文本模型则可以利用已经学到的语义关系，
使部分用词不同但含义相关的文本获得较接近的向量。

这一机制是本次效果改善的一种合理解释，
但实验没有单独验证各因素对提升的贡献。

不能把提升简单归因于维度降低：
两种方法的表示机制、预训练知识和文本处理方式都有差异，
低维表示本身不保证更好的聚类结果。

### 实验局限

1. 仅比较一个数据子集、一个预训练模型及一个随机种子。
2. 超过模型输入长度的文章会被截断，向量可能无法反映全文内容。
3. TF-IDF 存在 9 篇全零向量文档。
4. 两种表示的轮廓系数来自不同特征空间。
5. 当前评价反映这批文档的分组效果，不能直接代表新文档上的表现。

### 保存的实验结果

结果保存在 week4_text_results 文件夹中：

- sentence_embeddings.npz：文本向量、标签、原始文档索引及主要编码设置。
- clustering_comparison.csv：两种表示的聚类评价指标。
- tfidf_category_counts.csv：TF-IDF 簇与真实类别的交叉表。
- embedding_category_counts.csv：文本向量簇与真实类别的交叉表。
- cluster_assignments.csv：每篇文档的真实类别及两种聚类编号。

### 实验结论

在本次实验设置下，预训练文本向量的 ARI 和 NMI 均高于 TF-IDF，
图形学与太空文章也更加集中于各自的主要簇。
但两个宗教相关类别仍存在明显混合，聚类结果尚未完全对应真实类别。